In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import tensorflow as tf 
tf.debugging.disable_traceback_filtering()
import dl4ds as dds  # noqa: E402
tf.config.list_physical_devices('GPU')

In [ ]:
filepath = Path.home() / "atm_data" / "ERA5_northsea_original.nc"
ds = xr.open_dataset(filepath, engine="h5netcdf")  # without engine it fails

In [ ]:
y_train = y_test = y_val = np.expand_dims(ds.u10.values, axis=-1)[:, :36, :36, :]

In [ ]:
ds.u10.isel(valid_time=0).plot()

In [ ]:
y_train.shape

In [ ]:
ARCH_PARAMS = dict(n_filters=8,
                   n_blocks=8,
                   normalization=None,
                   dropout_rate=0.0,
                   dropout_variant='spatial',
                   attention=False,
                   activation='relu',
                   localcon_layer=True)

trainer = dds.SupervisedTrainer(
    backbone='resnet',
    upsampling='spc', 
    data_train=y_train, 
    data_val=y_val,
    data_test=y_test,
    data_train_lr=None, # here you can pass the LR dataset for training with explicit paired samples
    data_val_lr=None, # here you can pass the LR dataset for training with explicit paired samples
    data_test_lr=None, # here you can pass the LR dataset for training with explicit paired samples
    scale=6,
    time_window=None, 
    static_vars=None,
    predictors_train=None,
    predictors_val=None,
    predictors_test=None,
    interpolation='inter_area',
    patch_size=None, 
    batch_size=60, 
    loss='mae',
    epochs=10, 
    steps_per_epoch=None, 
    validation_steps=None, 
    test_steps=None, 
    learning_rate=(1e-3, 1e-4), lr_decay_after=1e4,
    early_stopping=False, patience=6, min_delta=0, 
    save=False, 
    save_path=None,
    show_plot=True, verbose=True, 
    device='GPU', 
    **ARCH_PARAMS)

In [ ]:
trainer.run()

# Inference

Let's evaluate the results on holdout data -- the test split of the benchmark dataset that has not been used while training (updating the network weights).

In [ ]:
pred = dds.Predictor(
    trainer, 
    y_test, 
    scale=6, 
    array_in_hr=True,
    static_vars=None, 
    predictors=None,
    time_window=None,
    interpolation='inter_area', 
    batch_size=8,
    scaler=None,
    save_path=None,
    save_fname=None,
    return_lr=True,
    device='CPU')

unscaled_y_pred, coarsened_array = pred.run()

Below is the coarsened version of the holdout, passed to the trained model for inference:

In [ ]:
ind = 100
ecv.plot((coarsened_array[ind][:,:,0], coarsened_array[ind][:,:,1]))

In [ ]:
unscaled_y_test = t2m_scaler_train.inverse_transform(y_test)

Finally, let's see compare the groundtruth HR t2m and the downscaled t2m obtained with DL4DS for a single time step. 

In [ ]:
ecv.plot((unscaled_y_test[ind].values, unscaled_y_pred[ind]), subplot_titles=('groundtruth t2m', 'downscaled t2m'))

Please take into account that we are using a dummy dataset with few samples, and are training in supervised fashion a single possible DL4DS architecture, without tuning hyperparameters. In order to find the best model, you should compute relevant metrics for your problem. This is out of the scope of this tutorial.